# Plant Disease Classification — Transfer Learning (MobileNetV2) + Gravitational Search Algorithm

Rebuilt to actually do what the title claims, and to run on **Google Colab** (GPU runtime).

- **Dataset:** PlantVillage only (the coffee-leaf class is excluded).
- **Model:** transfer learning on a pretrained **MobileNetV2** ImageNet backbone.
- **Hyperparameter tuning:** a genuine **Gravitational Search Algorithm (GSA)** that optimizes
  learning rate, dropout, and dense-layer width by validation accuracy.
- **Evaluation:** honest test metrics — accuracy, per-class precision/recall/F1, and a confusion matrix.

> Runtime: set **Runtime → Change runtime type → GPU** before running.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 1. Configuration

Transfer-learning heads converge fast, so the epoch counts are kept modest for Colab. The GSA
settings (population, iterations, epochs-per-candidate) are deliberately small so the search
finishes in a reasonable time — bump them up if you have runtime to spare.

In [ ]:
IMAGE_SIZE = 224          # MobileNetV2 native input size
BATCH_SIZE = 32
CHANNELS   = 3
SEED       = 42

# Final-training epochs (EarlyStopping usually stops earlier)
EPOCHS_HEAD     = 15      # train the new classifier head (backbone frozen)
EPOCHS_FINETUNE = 10      # fine-tune the top of the backbone

# GSA search budget (keep small for Colab)
GSA_AGENTS        = 5     # population size
GSA_ITERS         = 5     # iterations
GSA_EVAL_EPOCHS   = 3     # epochs per candidate evaluation
GSA_TRAIN_BATCHES = 60    # train subset (in batches) used while searching

## 2. Download the dataset from Roboflow

Uses the same pattern as the THESIS_2 `Computer_Vision` notebooks: the API key comes from
**Colab Secrets** (not uploaded or hardcoded). In Colab, open the 🔑 **Secrets** panel in the left
sidebar, add a secret named `ROBOFLOW_API_KEY`, and enable notebook access.

Because this is a **classification** model (not detection), we download the `"folder"` format, which
produces `train/`, `valid/`, and `test/` directories of class subfolders. Fill in your dataset's
`workspace` / `project` / `version` from its Roboflow download page.

In [ ]:
!pip install -q roboflow

import os
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
workspace = rf.workspace("plant-disease-detection-csu61")
project   = workspace.project("plant-disease-detection-iefbi")
version   = project.version(1)                 # <-- confirm the version number on the Download page
dataset   = version.download("folder")         # classification -> train/valid/test folders

DATA_DIR = dataset.location
print("Downloaded to:", DATA_DIR)
print("Contents:", os.listdir(DATA_DIR))

## 3. Load the pre-split data

Roboflow already splits the data, so we load `train/`, `valid/`, and `test/` directly instead of
splitting ourselves. Labels are integer-encoded (`label_mode="int"`) to pair with sparse
cross-entropy. If the export has no `test/` split, we carve one from the validation set.

In [ ]:
def load_split(name, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        os.path.join(DATA_DIR, name),
        labels="inferred",
        label_mode="int",
        shuffle=shuffle,
        seed=SEED,
        image_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
    )

train_ds = load_split("train", shuffle=True)
val_ds   = load_split("valid", shuffle=False)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)

if os.path.isdir(os.path.join(DATA_DIR, "test")):
    test_ds = load_split("test", shuffle=False)
else:
    n_test = max(len(val_ds) // 2, 1)          # no test split -> carve from validation
    test_ds = val_ds.take(n_test)
    val_ds  = val_ds.skip(n_test)
    print(f"No test/ split found; using {n_test} validation batches as test.")

print(NUM_CLASSES, "classes:")
CLASS_NAMES

In [ ]:
# PlantVillage only: refuse to proceed if a coffee (or other non-PlantVillage) folder slipped in.
coffee = [c for c in CLASS_NAMES if "coffee" in c.lower()]
assert not coffee, (
    f"Found non-PlantVillage class(es): {coffee}. "
    "Point DATA_DIR at a PlantVillage-only folder (remove the coffee_miner directory)."
)
print("Class set is PlantVillage-only. OK.")

In [ ]:
# Peek at a batch
plt.figure(figsize=(14, 14))
for image_batch, label_batch in train_ds.take(1):
    for i in range(12):
        plt.subplot(3, 4, i + 1)
        plt.imshow(image_batch[i].numpy().astype("uint8"))
        plt.title(CLASS_NAMES[label_batch[i]], fontsize=8)
        plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Prepare the input pipeline

Cache and prefetch so the GPU isn't waiting on disk. Only the training set is reshuffled each epoch.

In [ ]:
print(f"batches -> train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

## 5. Model builder (transfer learning)

MobileNetV2 pretrained on ImageNet, with the top removed. Its own preprocessing (`[0,255] → [-1,1]`)
is expressed as a `Rescaling` layer, and augmentation runs only during training. The backbone is
frozen while the new head trains; it is unfrozen later for fine-tuning.

`dense_units`, `dropout_rate`, and `learning_rate` are the knobs the GSA will optimize.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

def build_model(num_classes, dense_units, dropout_rate, learning_rate):
    tf.keras.backend.clear_session()
    base = MobileNetV2(
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS),
        include_top=False,
        weights="imagenet",
    )
    base.trainable = False

    inputs = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)     # MobileNetV2 preprocessing
    x = base(x, training=False)                         # keep BatchNorm in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(int(dense_units), activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model, base

## 6. Gravitational Search Algorithm (GSA)

A genuine implementation of the metaheuristic (Rashedi et al., 2009). Each **agent** is a candidate
hyperparameter vector in `[0,1]^3`, decoded to `(learning_rate, dropout, dense_units)`. Fitness is the
objective to **minimize**: `1 − validation_accuracy`.

Per iteration the algorithm:
1. evaluates every agent's fitness,
2. assigns **masses** (better solutions are heavier),
3. computes the **gravitational force** each agent feels from the *kbest* heaviest agents,
   with the gravitational constant `G(t)` decaying over time,
4. derives **acceleration → velocity → new position**.

`kbest` shrinks from all agents to 1, moving the search from exploration toward exploitation.

In [ ]:
# Search space: position in [0,1] -> real hyperparameters
def decode(pos):
    lr = 10.0 ** (-4.0 + 2.0 * pos[0])       # 1e-4 .. 1e-2 (log scale)
    dropout = 0.2 + 0.4 * pos[1]             # 0.2 .. 0.6
    units = int(round(64 + (512 - 64) * pos[2]))  # 64 .. 512
    return lr, dropout, units

def fitness(pos, train_ds, val_ds, epochs):
    lr, dropout, units = decode(pos)
    model, _ = build_model(NUM_CLASSES, units, dropout, lr)
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=0)
    val_acc = max(hist.history["val_accuracy"])
    del model
    tf.keras.backend.clear_session()
    return 1.0 - val_acc                     # minimize

In [ ]:
def gsa(train_ds, val_ds, dim=3, n_agents=GSA_AGENTS, max_iter=GSA_ITERS,
        epochs=GSA_EVAL_EPOCHS, G0=100.0, alpha=20.0, seed=SEED):
    rng = np.random.default_rng(seed)
    X = rng.random((n_agents, dim))          # positions
    V = np.zeros((n_agents, dim))            # velocities
    gbest_pos, gbest_fit = None, np.inf
    history = []

    for t in range(max_iter):
        fits = np.array([fitness(X[i], train_ds, val_ds, epochs) for i in range(n_agents)])

        best, worst = fits.min(), fits.max()
        i_best = int(fits.argmin())
        if fits[i_best] < gbest_fit:
            gbest_fit, gbest_pos = fits[i_best], X[i_best].copy()

        # Masses: best -> 1, worst -> 0 (minimization)
        if best == worst:
            M = np.ones(n_agents) / n_agents
        else:
            m = (fits - worst) / (best - worst)
            M = m / m.sum()

        G = G0 * np.exp(-alpha * t / max_iter)

        # kbest heaviest agents exert force; shrink from n_agents -> 1
        if max_iter > 1:
            kbest = int(round(n_agents - (n_agents - 1) * (t / (max_iter - 1))))
        else:
            kbest = n_agents
        kbest = max(kbest, 1)
        kset = np.argsort(fits)[:kbest]      # best (lowest fitness) first

        # Acceleration = sum of forces from kbest agents (agent's own mass cancels out)
        a = np.zeros((n_agents, dim))
        for i in range(n_agents):
            for j in kset:
                if j == i:
                    continue
                R = np.linalg.norm(X[i] - X[j]) + 1e-10
                a[i] += rng.random() * G * M[j] * (X[j] - X[i]) / R

        V = rng.random((n_agents, dim)) * V + a
        X = np.clip(X + V, 0.0, 1.0)

        history.append(gbest_fit)
        lr, dr, un = decode(gbest_pos)
        print(f"iter {t+1}/{max_iter}  best val_acc={1 - gbest_fit:.4f}  "
              f"(lr={lr:.2e}, dropout={dr:.2f}, units={un})  G={G:.2f}  kbest={kbest}")

    return gbest_pos, gbest_fit, history

## 7. Run the GSA search

The search evaluates candidates on a **subset** of the training data (for speed) but on the full
validation set. The winning hyperparameters are then used to train the full model.

In [ ]:
gsa_train_ds = train_ds.take(GSA_TRAIN_BATCHES)

best_pos, best_fit, gsa_history = gsa(gsa_train_ds, val_ds)
best_lr, best_dropout, best_units = decode(best_pos)

print("\nBest hyperparameters found by GSA:")
print(f"  learning_rate = {best_lr:.2e}")
print(f"  dropout       = {best_dropout:.2f}")
print(f"  dense_units   = {best_units}")
print(f"  val_accuracy  = {1 - best_fit:.4f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(gsa_history) + 1), [1 - f for f in gsa_history], marker="o")
plt.xlabel("GSA iteration")
plt.ylabel("Best validation accuracy")
plt.title("GSA convergence")
plt.grid(True, alpha=0.3)
plt.show()

## 8. Train the final model

**Phase 1** trains the new head with the backbone frozen. **Phase 2** unfreezes the top of
MobileNetV2 and fine-tunes at a lower learning rate. `EarlyStopping` restores the best weights.

In [ ]:
model, base = build_model(NUM_CLASSES, best_units, best_dropout, best_lr)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=3, restore_best_weights=True
)

print("Phase 1: training classifier head (backbone frozen)")
hist_head = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_HEAD, callbacks=[early_stop], verbose=1,
)

In [ ]:
print("Phase 2: fine-tuning top of MobileNetV2")
base.trainable = True
for layer in base.layers[:-30]:      # keep lower layers frozen
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=best_lr / 10.0),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

hist_ft = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_FINETUNE, callbacks=[early_stop], verbose=1,
)

In [ ]:
# Combine the two training phases for plotting
acc      = hist_head.history["accuracy"] + hist_ft.history["accuracy"]
val_acc  = hist_head.history["val_accuracy"] + hist_ft.history["val_accuracy"]
loss     = hist_head.history["loss"] + hist_ft.history["loss"]
val_loss = hist_head.history["val_loss"] + hist_ft.history["val_loss"]
split_at = len(hist_head.history["accuracy"])

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(acc, label="train")
plt.plot(val_acc, label="val")
plt.axvline(split_at - 0.5, color="gray", ls="--", label="fine-tune start")
plt.title("Accuracy"); plt.xlabel("epoch"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label="train")
plt.plot(val_loss, label="val")
plt.axvline(split_at - 0.5, color="gray", ls="--", label="fine-tune start")
plt.title("Loss"); plt.xlabel("epoch"); plt.legend()
plt.show()

## 9. Honest evaluation on the held-out test set

Overall accuracy plus a full per-class report and confusion matrix — this replaces the original
notebook's fabricated "accuracy × 2" metric.

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}   Test loss: {test_loss:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true, y_pred = [], []
for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(probs, axis=1))
y_true, y_pred = np.array(y_true), np.array(y_pred)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(11, 10))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
ticks = np.arange(NUM_CLASSES)
plt.xticks(ticks, CLASS_NAMES, rotation=90, fontsize=7)
plt.yticks(ticks, CLASS_NAMES, fontsize=7)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix")
thresh = cm.max() / 2.0
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=6,
                 color="white" if cm[i, j] > thresh else "black")
plt.tight_layout()
plt.show()

## 10. Save the model

In [ ]:
model.save("plant_disease_mobilenetv2_gsa.keras")
print("Saved to plant_disease_mobilenetv2_gsa.keras")